# Ham ya da Spam?

🎯 Bu görevin amacı, e-postaları **spam (1)** veya **normal e-posta (0)** olarak sınıflandırmaktır.

🧹 İlk olarak, bu metin verilerine **temizleme (cleaning)** teknikleri uygulanacaktır.

👩🏻‍🔬 Ardından, temizlenmiş metinler **sayısal bir gösterime** dönüştürülecektir.

✉️ Son olarak, her bir e-postayı spam mı yoksa normal mi olduğunu sınıflandırmak için  
***Multinomial Naive Bayes*** modeli uygulanacaktır.


## (0) NTLK kütüphanesi (Doğal Dil Araç Seti)

In [1]:
!pip install nltk


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
# nltk'yi ilk kez içe aktarırken, birkaç yerleşik kütüphaneyi de indirmemiz gerekir.

import nltk

nltk.download('stopwords')
nltk.download('punkt')      # nltk<3.9.0 için
nltk.download('punkt_tab')  # nltk>=3.9.0 için
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/fukansimsek/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to
[nltk_data]     /home/fukansimsek/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/fukansimsek/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/fukansimsek/nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /home/fukansimsek/nltk_data...


True

In [3]:
import pandas as pd

df = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/ham_spam_emails.csv")
df.head()

,text,spam
0,Subject: naturally irresistible your corporate...,1
1,Subject: the stock trading gunslinger fanny i...,1
2,Subject: unbelievable new homes made easy im ...,1
3,Subject: 4 color printing special request add...,1
4,"Subject: do not have money , get software cds ...",1


## (1) (Metin) veri setinin temizlenmesi

Veri kümesi, ham [0] veya spam [1] olarak sınıflandırılan e-postalardan oluşur. Tahmin modelini eğitmeden önce veri kümesini temizlemeniz gerekir.

### (1.1) Noktalama İşaretlerini Kaldır

❓ Noktalama işaretlerini kaldırmak için bir işlev oluşturun. Bunu `text` sütununa uygulayın ve çıktıyı `clean_text` adlı veri çerçevesinin yeni bir sütununa ekleyin. ❓

In [16]:
data= {'text': ["Merhaba, dünya!", "Python; veri analizi için harikadır.", "Noktalama? İşaretleri... 123Gitsin!"]}
df = pd.DataFrame(data)

df['clean_text'] = df['text'].str.replace(f'[{string.punctuation}]', '', regex=True)

print(df)

                                   text                          clean_text
0                       Merhaba, dünya!                       Merhaba dünya
1  Python; veri analizi için harikadır.  Python veri analizi için harikadır
2   Noktalama? İşaretleri... 123Gitsin!      Noktalama İşaretleri 123Gitsin


### (1.2) Küçük Harf

❓ Metni küçük harfe çeviren bir işlev oluşturun. Bunu `clean_text`'e uygulayın ❓

In [17]:
df['clean_text'] = df['clean_text'].str.lower()

print(df[['text', 'clean_text']].head())

                                   text                          clean_text
0                       Merhaba, dünya!                       merhaba dünya
1  Python; veri analizi için harikadır.  python veri analizi için harikadır
2   Noktalama? İşaretleri... 123Gitsin!     noktalama i̇şaretleri 123gitsin


### (1.3) Sayıları Kaldır

❓ Metinden sayıları kaldırmak için bir işlev oluşturun. Bunu `clean_text`'e uygulayın ❓

In [18]:
df['clean_text'] = df['clean_text'].str.replace(r'\d+', '', regex=True)

print(df)

                                   text                          clean_text
0                       Merhaba, dünya!                       merhaba dünya
1  Python; veri analizi için harikadır.  python veri analizi için harikadır
2   Noktalama? İşaretleri... 123Gitsin!        noktalama i̇şaretleri gitsin


### (1.4) StopWords'ü kaldırın

❓ Metinden durdurma kelimelerini kaldırmak için bir işlev oluşturun. Bunu `clean_text`'e uygulayın. ❓

In [20]:
from nltk.corpus import stopwords

stop_words_list = set(stopwords.words('turkish'))

def remove_stopwords(text):

    words = text.split()

    filtered_words = [word for word in words if word not in stop_words_list]

    return " ".join(filtered_words)

df['clean_text'] = df['clean_text'].apply(remove_stopwords)

print(df['clean_text'].head())


0                    merhaba dünya
1    python veri analizi harikadır
2     noktalama i̇şaretleri gitsin
Name: clean_text, dtype: object


### (1.5) Lemmatize

❓ Metni lemmatize etmek için bir fonksiyon oluşturun. Çıktının bir kelime listesi değil, tek bir dize olduğundan emin olun. Bunu `clean_text`'e uygulayın. ❓

In [22]:
!pip install zeyrek

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 931.0/931.0 kB 8.6 MB/s  0:00:00

[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [23]:
import zeyrek

analyzer = zeyrek.MorphAnalyzer()

def lemmatize_text(text):

    words = text.split()
    lemmatized_words = []

    for word in words:

        analysis = analyzer.lemmatize(word)

        if analysis:

            lemma = analysis[0][1][0]
            lemmatized_words.append(lemma)
        else:
            lemmatized_words.append(word)

    return " ".join(lemmatized_words)

df['clean_text'] = df['clean_text'].apply(lemmatize_text)

print(df['clean_text'].head())

APPENDING RESULT: <(merhaba_Interj)(-)(merhaba:interjRoot_ST)>
APPENDING RESULT: <(merhaba_Noun)(-)(merhaba:noun_S + a3sg_S + pnon_S + nom_ST)>
APPENDING RESULT: <(dünya_Noun)(-)(dünya:noun_S + a3sg_S + pnon_S + nom_ST)>
APPENDING RESULT: <(Dünya_Noun_Prop)(-)(dünya:nounProper_S + a3sg_S + pnon_S + nom_ST)>
APPENDING RESULT: <(Python_Noun_Prop)(-)(python:nounProper_S + a3sg_S + pnon_S + nom_ST)>
APPENDING RESULT: <(veri_Noun)(-)(veri:noun_S + a3sg_S + pnon_S + nom_ST)>
APPENDING RESULT: <(analiz_Noun)(-)(analiz:noun_S + a3sg_S + pnon_S + i:acc_ST)>
APPENDING RESULT: <(analiz_Noun)(-)(analiz:noun_S + a3sg_S + i:p3sg_S + nom_ST)>
APPENDING RESULT: <(harika_Adj)(-)(harika:adjectiveRoot_ST + adjZeroDeriv_S + nVerb_S + nPresent_S + nA3sg_S + dır:nCop_ST)>
APPENDING RESULT: <(Harika_Noun_Prop)(-)(harika:nounProper_S + a3sg_S + pnon_S + nom_ST + nounZeroDeriv_S + nVerb_S + nPresent_S + nA3sg_S + dır:nCop_ST)>
APPENDING RESULT: <(noktalamak_Verb)(-)(noktala:verbRoot_S + ma:vNeg_S + vImp_S + vA

0                    merhaba Dünya
1        Python veri analiz Harika
2    noktalamak i̇şaretleri gitmek
Name: clean_text, dtype: object


## (2) Bag-of-Words Modellemesi

### (2.1) Metin verilerini sayılara dönüştürme

❓ `clean_text`'i varsayılan CountVectorizer ile Bag-of-Words temsiline vektörleştirin. `X_bow` olarak kaydedin. ❓

In [24]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()

X_bow = vectorizer.fit_transform(df['clean_text'])

print(f"Vektör Matris Boyutu: {X_bow.shape}")

print("Örnek Özellikler (Kelimeler):", vectorizer.get_feature_names_out()[:10])

Vektör Matris Boyutu: (3, 9)
Örnek Özellikler (Kelimeler): ['analiz' 'dünya' 'gitmek' 'harika' 'merhaba' 'noktalamak' 'python' 'veri'
 'şaretleri']


### (2.2) Çok terimli Naive Bayes Modellemesi

❓ MultinomialNB modelini bag-of-words verileriyle çapraz doğrulayın. Modelin doğruluğunu puanlayın. ❓

In [25]:
import numpy as np
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder

data = {
    'clean_text': [
        'film harika mutalaka izleyin', 'berbat bir çekim zaman kaybı',
        'bayıldım çok etkileyici', 'hiç beğenmedim tavsiye etmem',
        'senaryo çok zayıf sıkıcı', 'mükemmel bir oyunculuk tebrikler',
        'izlediğim en kötü şey', 'tek kelimeyle muazzam'
    ],
    'sentiment': ['pozitif', 'negatif', 'pozitif', 'negatif', 'negatif', 'pozitif', 'negatif', 'pozitif']
}
df = pd.DataFrame(data)

le = LabelEncoder()
y = le.fit_transform(df['sentiment'])

vectorizer = CountVectorizer()
X_bow = vectorizer.fit_transform(df['clean_text'])

nb_model = MultinomialNB()

scores = cross_val_score(nb_model, X_bow, y, cv=3, scoring='accuracy')

print(f"Tanımlanan Etiketler (y): {y}")
print(f"Kategori Eşleşmesi: {dict(zip(le.classes_, le.transform(le.classes_)))}")
print(f"Ortalama Doğruluk Skoru: %{scores.mean() * 100:.2f}")

Tanımlanan Etiketler (y): [1 0 1 0 0 1 0 1]
Kategori Eşleşmesi: {'negatif': 0, 'pozitif': 1}
Ortalama Doğruluk Skoru: %27.78


🏁 Tebrikler!

💾 Not defterinizi git add/commit/push yapmayı unutmayın...

🚀 ... ve bir sonraki challenge'a geçin!